# 09 — Statistical Significance Testing (M7)

Runs `scripts/evaluate_all.py`: paired bootstrap significance testing (`src/biomedical_ir/statistics.py`, n_resamples=10000, seed=42) at the query level, for the 5 comparisons Section 16 of the project spec names explicitly plus TF-IDF vs. BM25 (added to directly resolve H1). 6 comparisons × 5 primary metrics = 30 tests.

**Prerequisite:** requires all six `results/runs/*.json` (notebooks 02–07).

In [ ]:
# If running on Colab, clone the repo and install deps. Skipped automatically
# when already inside a local checkout (REPO_ROOT / 'src' already importable).
import os, sys, subprocess

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    if not os.path.exists('biomedical-hybrid-ir'):
        subprocess.run(['git', 'clone', 'https://github.com/Arungharami/biomedical-hybrid-ir'], check=True)
    os.chdir('biomedical-hybrid-ir')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.'], check=True)

sys.path.insert(0, os.path.join(os.getcwd(), 'src'))
print('cwd:', os.getcwd())

In [ ]:
import time
start = time.perf_counter()
result = subprocess.run([sys.executable, 'scripts/evaluate_all.py'], cwd=os.getcwd())
print(f'\nexit code: {result.returncode}, elapsed: {time.perf_counter()-start:.1f}s')
assert result.returncode == 0, 'Script failed -- see output above.'

## Full results table

In [ ]:
import json
import pandas as pd

stats = json.load(open('results/tables/statistical_tests.json'))
df = pd.DataFrame(stats['comparisons'])
df['comparison'] = df['model_a'] + ' vs ' + df['model_b']
df[['comparison', 'metric', 'mean_diff', 'p_value', 'significant_at_alpha_0.05']]

## Per-hypothesis verdicts

- **H1 — REJECTED.** TF-IDF significantly beats BM25 (P@10 p=0.017, Recall@100 p=0.010, nDCG@10 p=0.030).
- **H2 — NOT SUPPORTED.** BGE vs. MedCPT: no significant difference (all p≥0.12).
- **H3 — NOT SUPPORTED** for the comparison tested. MedCPT significantly beats Hybrid RRF on Recall@100 (p=0.040), opposite the predicted direction.
- **H4 — PARTIALLY SUPPORTED.** Latency increase unambiguous; nDCG@10 improvement is the best point estimate in the study but not significant (p=0.194); P@10 improves significantly (p=0.015).

**Most robust finding:** both dense retrievers significantly outperform BM25 on every primary metric (p<0.005 each). See `paper/methodology.md` and `paper/results.md` §9 for the full writeup.